# 🏊 05 · 矿池 & 分红机制

> Stratum 协议、Share、PPS vs PPLNS。

## 5.1 为什么需要矿池？

**个体矿工的问题：**
- 收益方差巨大 — 可能 3 年颗粒无收，也可能 3 天中 2 个块
- 现金流不稳定 — 电费每天要交，收益不确定
- 运营成本 — 需要维护全节点、7/24 监控

**矿池的解决方案：**
- 矿工抱团，收益按贡献平滑分配
- 矿池运营基础设施，矿工只需接收任务 + 提交结果
- 目前 **99%+** 的比特币算力通过矿池组织

## 5.2 Stratum 协议

矿池和矿工之间通过 **Stratum** (JSON-RPC over TCP) 通信：

```
矿工 → 矿池: mining.subscribe
矿池 → 矿工: mining.set_difficulty (share 难度)
矿池 → 矿工: mining.notify (新任务)
矿工 → 矿池: mining.submit (提交 share)
```

关键：矿池设的 **share 难度** 远低于网络难度，
这样矿工能频繁提交 share，矿池可以验证矿工确实在干活。

In [ ]:
import hashlib, time, random
from dataclasses import dataclass, field
from typing import List, Dict
from collections import defaultdict

@dataclass
class Share:
    miner_name: str
    nonce: int
    hash: str
    difficulty: float

class MiningPool:
    def __init__(self, name: str, block_reward: float = 3.125):
        self.name = name
        self.block_reward = block_reward
        self.shares: List[Share] = []
        self.stats: Dict[str, dict] = defaultdict(lambda: {"shares": 0, "earned": 0.0})
        self.total_blocks = 0

    def submit_share(self, miner_name: str, share: Share) -> bool:
        # Verify share
        h_int = int(share.hash, 16)
        share_target = (2**256) / share.difficulty
        if h_int >= share_target:
            return False

        self.shares.append(share)
        self.stats[miner_name]["shares"] += 1

        # Check if this also satisfies network difficulty
        network_target = (2**256) / 70_000_000_000_000
        if h_int < network_target:
            self.total_blocks += 1
            self.distribute_reward(method="pplns", n=100)
            return True
        return False

    def distribute_reward(self, method="pplns", n=100):
        # PPLNS: Pay Per Last N Shares
        recent = self.shares[-min(n, len(self.shares)):]
        for share in recent:
            self.stats[share.miner_name]["earned"] += self.block_reward / len(recent)

class PoolMiner:
    def __init__(self, name: str, pool: MiningPool, hashrate: int):
        self.name = name
        self.pool = pool
        self.hashrate = hashrate
        self.total_hashes = 0

    def mine_step(self, duration: float = 0.1):
        hashes = int(self.hashrate * duration)
        share_diff = self.pool.shares[-1].difficulty if self.pool.shares else 1000
        share_target = int((2**256) / share_diff)

        for _ in range(hashes):
            nonce = random.randint(0, 2**32 - 1)
            h = hashlib.sha256(hashlib.sha256(
                f"block|{self.name}|{nonce}|{time.time()}".encode()
            ).digest()).hexdigest()
            self.total_hashes += 1

            h_int = int(h, 16)
            if h_int < share_target:
                share = Share(self.name, nonce, h, share_diff)
                found_block = self.pool.submit_share(self.name, share)
                if found_block:
                    print(f"   🎉 {self.name} found a block!")
                break
print("✅ Pool model ready")


In [ ]:
# Simulate a mining pool
pool = MiningPool("Lobster Pool 🦞", block_reward=3.125)

miners = [
    PoolMiner("Your MacBook", pool, hashrate=10_000),
    PoolMiner("Gaming Laptop", pool, hashrate=50_000),
    PoolMiner("RTX 4090 Rig", pool, hashrate=3_000_000_000),
    PoolMiner("S19 #1", pool, hashrate=110_000_000_000_000),
    PoolMiner("S19 #2", pool, hashrate=110_000_000_000_000),
    PoolMiner("S19 #3", pool, hashrate=110_000_000_000_000),
]

print(f"🦞 {pool.name} — {len(miners)} miners")
total_hr = sum(m.hashrate for m in miners)
print(f"   Total hashrate: {total_hr:,.0f} H/s ({total_hr/1e12:.1f} TH/s)")
print()

for round_num in range(30):
    for miner in miners:
        miner.mine_step(duration=0.03)

print("\n📊 Pool Stats:")
print("-" * 50)
total_shares = sum(s["shares"] for s in pool.stats.values())
print(f"{'Miner':<16} | {'Shares':>7} | {'Share %':>7} | {'Earned (BTC)':>12}")
print("-" * 50)
for name in sorted(pool.stats, key=lambda n: pool.stats[n]["shares"], reverse=True):
    s = pool.stats[name]
    pct = s["shares"] / total_shares * 100 if total_shares else 0
    print(f"{name:<16} | {s['shares']:>7} | {pct:>6.1f}% | {s['earned']:>12.8f}")

print(f"\nYour MacBook contributed {pool.stats.get('Your MacBook', {}).get('shares', 0)} shares")
print("How much of the block reward would you get? Almost nothing!")
print("That's why pool mining doesn't help small miners — you need real hardware.")


## 5.3 分红模式对比

| 模式 | 原理 | 风险承担 | 适用 |
|------|------|----------|------|
| **PPS** | 每 share 固定支付 | 矿池承担 | 求稳的小矿工 |
| **PPLNS** | 挖到块时按最后 N 个 share 分 | 矿工共担 | 忠实矿工 |
| **FPPS** | PPS + 手续费分成 | 矿池承担 | 目前主流 |
| **SOLO** | 挖到全拿，挖不到零 | 矿工全担 | 信仰充值 |

矿池抽水通常 1-3%。

## 小结

✅ Stratum = JSON-RPC over TCP，矿池标准协议
✅ Share 难度 << 网络难度，保证频繁验证
✅ PPLNS / FPPS 平滑矿工收益
✅ 矿池解决了方差问题，但导致算力集中化
✅ 散户进矿池照样赚不到钱——算力占比太低

**下一课**: 难度调整——比特币的恒温器。